In [ ]:
%%bash
# v3: the v2 hybrid trained on asl-signs train + ASL Citizen train (mapped signs) with webcam
# augmentation, three seeds, selected and calibrated on validation only, exported as fp16 ONNX.
# No test split is read here: the asl-signs and ASL Citizen test sets and test_clips_v2 are
# evaluated once, locally (kaggle/ASL_CITIZEN.md).
#
# Settings: Accelerator GPU (T4), Internet On, Save Version > Save & Run All (Commit).
# Inputs: Competitions > Google - Isolated Sign Language Recognition; Your Work > Datasets >
# asl-citizen-landmarks-private; and, unless this branch is pushed to GitHub, your private dataset
# edge-vision-src holding edge-vision-src.tar.gz (git archive of the commit to train).
# Resumable: finished seeds are skipped when an earlier version's output is added as an input.
# Licence: everything stays private (docs/DATA_LICENSES.md). Generated by
# scripts/build_kaggle_notebooks.py from kaggle/train_v3.sh.
set -u
WORK=/kaggle/working
LOG="$WORK/train_v3.log"
REPO="$WORK/edge-vision"
BRANCH="main"            # used only when no edge-vision-src archive is attached
MAX_HOURS=9              # no new seed starts after this, so export and packing fit in 12 h
START=$(date +%s)
mkdir -p "$WORK"
{
  echo "=== $(date -u '+%F %T') UTC: start"
  if [ ! -d "$REPO/src/edgevision" ]; then
    ARCHIVE=$(find /kaggle/input -maxdepth 4 -name 'edge-vision-src*.tar.gz' 2>/dev/null | head -1)
    if [ -n "$ARCHIVE" ]; then
      echo "source: $ARCHIVE"
      mkdir -p "$REPO" && tar -xzf "$ARCHIVE" -C "$REPO"
    else
      echo "source: git clone -b $BRANCH"
      git clone --depth 1 -b "$BRANCH" https://github.com/Swapnil007-Curious/edge-vision.git "$REPO"
    fi
  fi
  cd "$REPO" || exit 0
  pip install -q --no-deps -e . || echo "ERROR: pip install of the repository failed"
  python -c "import onnx, onnxruntime" 2>/dev/null || pip install -q onnx onnxruntime
  python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); print((torch.ones(4, device='cuda') * 2).sum().item())" \
    || echo "WARNING: no usable GPU; training on the CPU is far too slow"

  # Carry finished runs over from an earlier version of this notebook, if one is attached.
  for previous in $(find /kaggle/input -maxdepth 5 -type d -name e_hybrid_v3citizen 2>/dev/null); do
    mkdir -p "$WORK/runs" && cp -rn "$previous" "$WORK/runs/" && echo "carried over runs from $previous"
  done

  DATA_ROOT=$(dirname "$(find /kaggle/input -maxdepth 4 -name train.csv -path '*asl-signs*' 2>/dev/null | head -1)")
  LANDMARKS=$(dirname "$(find /kaggle/input -maxdepth 6 -name manifest.csv -path '*asl-citizen*' 2>/dev/null | head -1)")
  echo "asl-signs: $DATA_ROOT"
  echo "ASL Citizen landmarks: $LANDMARKS"
  if [ ! -f "$WORK/processed_v2/features.npz" ]; then
    python scripts/preprocess.py --config experiments/preprocess_v2.yaml --data-root "$DATA_ROOT" --output-dir "$WORK/processed_v2"
  fi
  # The asl-signs split must hold the participants of the shipped v2 model, so v2 and v3 are
  # compared on the same test sequences.
  python -c "import json; a = json.load(open('$WORK/processed_v2/splits_signer_independent.json'))['participants']; b = json.load(open('results/runs/cnn_signer_independent/seed_0/splits_signer_independent.json'))['participants']; assert a == b, 'split differs from v1/v2'; print('asl-signs split: same participants as v1/v2')" \
    || { echo "ERROR: the asl-signs split differs; stopping"; exit 0; }
  if [ ! -f "$WORK/processed_v3/features.npz" ]; then
    python scripts/v3_kaggle.py build-dataset --landmarks "$LANDMARKS" --kaggle-processed "$WORK/processed_v2" --out "$WORK/processed_v3"
  fi

  for SEED in 0 1 2; do
    if [ -f "$WORK/runs/e_hybrid_v3citizen/seed_$SEED/summary.json" ]; then
      echo "seed $SEED: done"
      continue
    fi
    ELAPSED=$(( ($(date +%s) - START) / 3600 ))
    if [ "$ELAPSED" -ge "$MAX_HOURS" ]; then
      echo "seed $SEED: not started after $ELAPSED h; save this version and rerun with its output attached"
      continue
    fi
    python scripts/train.py --config experiments/train_e_hybrid_v3citizen.yaml --seed "$SEED" \
      --processed-dir "$WORK/processed_v3" --output-root "$WORK/runs"
  done

  if CHOSEN=$(python scripts/v3_kaggle.py select --runs "$WORK/runs/e_hybrid_v3citizen" --out "$WORK/v3_export/selection.json"); then
    echo "chosen on validation: $CHOSEN"
    python scripts/export_web.py --run "$CHOSEN" --processed-dir "$WORK/processed_v3" --variant fp16 \
      --data-source asl-signs+asl-citizen --out "$WORK/v3_export/model" \
      --golden-dir "$WORK/v3_export/golden" --work-dir "$WORK/v3_export/work"
  fi
  echo "=== $(date -u '+%F %T') UTC: packing"
} 2>&1 | tee -a "$LOG"
cd "$REPO" 2>/dev/null && python scripts/v3_kaggle.py pack --work "$WORK" --output "$WORK/v3_bundle.zip" 2>&1 | tee -a "$LOG"
echo "=== $(date -u '+%F %T') UTC: finished; this cell always exits 0, so read $LOG" | tee -a "$LOG"
exit 0
